In [ ]:
import os
import h5py
import numpy as np
import time
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support

# Define the TCGA Folder Path
base_path = "/Users/franciscodowney/Desktop/UCLA/Courses/Winter Quarter 2025/228/Project/Final Project/tcga/tcga/"
luad_path = os.path.join(base_path, "luad")
lusc_path = os.path.join(base_path, "lusc")

# Function to get all .h5 files recursively
def get_h5_files(root_folder):
    h5_files = []
    for subdir, _, files in os.walk(root_folder):
        for file in files:
            if file.endswith(".h5"):
                h5_files.append(os.path.join(subdir, file))
    return h5_files

# Get LUAD and LUSC .h5 files
luad_files = get_h5_files(luad_path)
lusc_files = get_h5_files(lusc_path)

# Split into train-test
train_luad, test_luad = train_test_split(luad_files, test_size=0.2, random_state=42)
train_lusc, test_lusc = train_test_split(lusc_files, test_size=0.2, random_state=42)
train_files = train_luad + train_lusc
test_files = test_luad + test_lusc

# Function to stream features one file at a time
def stream_features(file_list):
    for file_path in file_list:
        # Small delay to ease file system load
        time.sleep(0.1)
        with h5py.File(file_path, "r") as h5_file:
            if "features" in h5_file:
                features = h5_file["features"][:]
                # Infer label from file path (adjust case as needed)
                label = 0 if "luad" in file_path.lower() else 1
                labels = np.full(features.shape[0], label)
                yield features, labels

# Initialize scaler and MLP model
scaler = StandardScaler()
mlp_clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=10, warm_start=True, random_state=42)

# Incremental training using buffering to ensure both classes are present in each batch
first_batch = True
buffer_X = []
buffer_y = []

for X_batch, y_batch in stream_features(train_files):
    buffer_X.append(X_batch)
    buffer_y.append(y_batch)
    combined_X = np.vstack(buffer_X)
    combined_y = np.concatenate(buffer_y)
    # Proceed only if both classes are present in the buffered data
    if set(np.unique(combined_y)) == {0, 1}:
        X_combined_scaled = scaler.partial_fit(combined_X).transform(combined_X)
        if first_batch:
            mlp_clf.partial_fit(X_combined_scaled, combined_y, classes=np.array([0, 1]))
            first_batch = False
        else:
            mlp_clf.partial_fit(X_combined_scaled, combined_y)
        # Clear the buffer after training on the combined batch
        buffer_X = []
        buffer_y = []

# Evaluate the model on the test set
y_true, y_pred, y_prob = [], [], []
for X_batch, y_batch in stream_features(test_files):
    X_batch_scaled = scaler.transform(X_batch)
    y_pred.extend(mlp_clf.predict(X_batch_scaled))
    y_prob.extend(mlp_clf.predict_proba(X_batch_scaled)[:, 1])
    y_true.extend(y_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

accuracy = accuracy_score(y_true, y_pred)
auroc = roc_auc_score(y_true, y_prob)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

print("✅ MLP Model (Streaming) Trained Successfully!")
print(f"🔹 Accuracy: {accuracy:.4f}")
print(f"🔹 AUROC Score: {auroc:.4f}")
print(f"🔹 Precision: {precision:.4f}")
print(f"🔹 Recall: {recall:.4f}")
print(f"🔹 F1 Score: {f1:.4f}")
